# 04 · Tiny decoder-only Transformer from scratch (P5)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/04_tiny_transformer.ipynb)

A GPT-style pre-LN Transformer (`src/cegqa/tiny/model.py`) with a byte-level BPE tokenizer (8k vocab)
trained on SQuAD in the format `<ctx> passage <q> question <a> answer <eos>`.
Three sizes (≈1M, 4M and 17M parameters) are trained to 20M tokens each; validation cross-entropy
is measured at log-spaced token counts (all tokens and answer tokens only), and greedy EM/F1 is
compared against Qwen3-0.6B. Runtime on a T4: ~25 min.

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
GPU_PROFILE = "auto"  #@param ["auto", "t4", "a100"]
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
if GPU_PROFILE == "auto":
    GPU_PROFILE = "a100" if any(k in GPU for k in ("A100", "H100")) else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"  # different budgets/sizes: keep results of the two profiles apart
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} | profile: {GPU_PROFILE} | outputs -> {OUT}")
print("configs:", CONFIG_FILES)

In [ ]:
!python -m cegqa.cli prepare $CONFIG

In [ ]:
!python -m cegqa.cli tiny $CONFIG

In [ ]:
from cegqa.utils import load_config
cfg = load_config(CONFIG_FILES)

In [ ]:
import json, pandas as pd
s = json.load(open(f"{cfg['output_dir']}/tiny/summary.json"))
pd.DataFrame([{"model": x["name"], "params (M)": round(x["params"]["total"] / 1e6, 2),
               "non-emb (M)": round(x["params"]["non_embedding"] / 1e6, 2), "tokens (M)": round(x["tokens"] / 1e6, 1),
               "tokens/param": round(x["tokens_per_param"], 1), "FLOPs": f"{x['est_train_flops']:.2e}",
               "minutes": round(x["train_time_s"] / 60, 1), "val CE": round(x["final_val_ce"], 3),
               "answer CE": round(x["final_val_answer_ce"], 3), "EM": round(x["em"], 1), "F1": round(x["f1"], 1)} for x in s])

In [ ]:
!python -m cegqa.cli report $CONFIG
from IPython.display import Image
Image(f"{cfg['output_dir']}/figures/tiny_ce_curves.png")

In [ ]:
# sample predictions from the largest model
from cegqa.utils import read_jsonl
name = cfg["tiny"]["models"][-1]["name"]
pd.DataFrame(read_jsonl(f"{cfg['output_dir']}/tiny/{name}/predictions_{cfg['tiny']['eval_source']}.jsonl")[:15])[["question", "answers", "prediction", "f1"]]